In [ ]:
##测试版，已废弃
##创建meta.json和report.txt
##会在root_dir的每个子文件夹中创建这两个文件
from pathlib import Path

root_dir = Path(r"G:\wxy_codes\Fetal_hydronephrosis\fetal_hydro_dataset\raw\case_0001\新建文件夹")

for sub_dir in root_dir.iterdir():
    if not sub_dir.is_dir():
        continue

    meta_path = sub_dir / "meta.json"
    report_path = sub_dir / "report.txt"

    if not meta_path.exists():
        meta_path.write_text("{}", encoding="utf-8")
        print(f"已创建: {meta_path}")

    if not report_path.exists():
        report_path.write_text("", encoding="utf-8")
        print(f"已创建: {report_path}")

print("完成")

In [ ]:
##升级版
##创建meta.json和report.txt
##会在root_dir的每个子文件夹的子文件夹的子文件夹中创建这两个文件

# 比如 以G:\wxy_codes\Fetal_hydronephrosis\fetal_hydro_dataset\raw
# 为根目录，遍历：

# 第一层：case_0001、case_0002 …
# 第二层：1prenatal、2postnatal、3postoperative
# 第三层：例如 20240307003299

# 然后在第三层文件夹里创建：
# meta.json
# report.txt

#如果已经存在，就不创建直接跳过


from pathlib import Path

##G:\wxy_codes\Fetal_hydronephrosis\第二批数据及处理过程\batch2data_2\raw
root_dir = Path(r"G:\wxy_codes\Fetal_hydronephrosis\第二批数据及处理过程\batch2data_2\raw")

for case_dir in root_dir.iterdir():
    if not case_dir.is_dir():
        continue

    for stage_dir in case_dir.iterdir():
        if not stage_dir.is_dir():
            continue

        for visit_dir in stage_dir.iterdir():
            if not visit_dir.is_dir():
                continue

            meta_path = visit_dir / "meta.json"
            report_path = visit_dir / "report.txt"

            if not meta_path.exists():
                meta_path.write_text("{}", encoding="utf-8")
                print(f"已创建: {meta_path}")
            else:
                print(f"已存在，跳过: {meta_path}")

            if not report_path.exists():
                report_path.write_text("", encoding="utf-8")
                print(f"已创建: {report_path}")
            else:
                print(f"已存在，跳过: {report_path}")

print("完成")

In [ ]:
##测试版，已废弃
##向meta.json里填入内容 
##孕天和report.txt需要之后自己填写
##会在root_dir的每个子文件夹中的meta.json写入内容
from pathlib import Path
import json

root_dir = Path(r"G:\wxy_codes\Fetal_hydronephrosis\fetal_hydro_dataset\raw\case_0001\新建文件夹")

created_meta_count = 0
created_report_count = 0
updated_meta_count = 0

for sub_dir in root_dir.iterdir():
    if not sub_dir.is_dir():
        continue

    visit_id = sub_dir.name

    # 从 visit_id 前8位提取日期
    if len(visit_id) >= 8 and visit_id[:8].isdigit():
        exam_date = f"{visit_id[:4]}-{visit_id[4:6]}-{visit_id[6:8]}"
    else:
        exam_date = ""

    meta_data = {
        "visit_id": visit_id,
        "phase": "postnatal",
        "gestational_age_days": 0,
        "exam_date": exam_date
    }

    meta_path = sub_dir / "meta.json"
    report_path = sub_dir / "report.txt"

    # 写入 meta.json
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(meta_data, f, ensure_ascii=False, indent=2)
    print(f"已写入: {meta_path}")
    updated_meta_count += 1

    # 创建空的 report.txt（如果不存在）
    if not report_path.exists():
        report_path.write_text("", encoding="utf-8")
        print(f"已创建: {report_path}")
        created_report_count += 1

print("\n处理完成")
print(f"写入 meta.json 数量: {updated_meta_count}")
print(f"新建 report.txt 数量: {created_report_count}")

In [ ]:
##上面代码的升级版

# 以G:\wxy_codes\Fetal_hydronephrosis\fetal_hydro_dataset\raw
# 为根目录，遍历到第三级文件夹，例如：

# G:\wxy_codes\Fetal_hydronephrosis\fetal_hydro_dataset\raw\case_0002\1prenatal\20240307003299

# 然后处理其中的：

#     meta.json
#     report.txt

# 规则如下：

# 1.只处理这一级目录下的 meta.json 和 report.txt
# 2.如果文件不存在，则创建
# 3.如果文件已存在但有内容，则跳过，不覆盖
# 4.visit_id 取当前文件夹名，例如 20240307003299
# 5.exam_date 取 visit_id 前 8 位，格式化成 YYYY-MM-DD
# 6.phase 自动取当前目录的上一级文件夹名，并去掉前面的数字前缀
#     例如：
#     1prenatal → prenatal
#     2postnatal → postnatal
#     3postoperative → postoperative


from pathlib import Path
import json
import re

root_dir = Path(r"G:\wxy_codes\Fetal_hydronephrosis\第二批数据及处理过程\batch2data_2\raw")

created_meta_count = 0
created_report_count = 0
skipped_meta_count = 0
skipped_report_count = 0


def is_empty_meta_json(file_path: Path) -> bool:
    """
    判断 meta.json 是否为空：
    1. 文件不存在 -> 视为空
    2. 文件内容为空字符串 -> 视为空
    3. 文件内容为 {} -> 视为空
    4. 其他有实际字段内容 -> 视为非空
    """
    if not file_path.exists():
        return True

    try:
        content = file_path.read_text(encoding="utf-8").strip()
        if content == "":
            return True

        # 如果是合法 json，进一步判断是否为空对象
        data = json.loads(content)
        if data == {}:
            return True

        return False
    except Exception:
        # 如果不是合法 json，但文件里有内容，为安全起见视为非空，避免误覆盖
        return False


def is_empty_text_file(file_path: Path) -> bool:
    """
    判断 report.txt 是否为空：
    1. 文件不存在 -> 视为空
    2. 文件内容去掉空白后为空 -> 视为空
    3. 否则视为非空
    """
    if not file_path.exists():
        return True

    try:
        content = file_path.read_text(encoding="utf-8")
        return content.strip() == ""
    except Exception:
        # 读失败时，为安全起见视为非空
        return False


for case_dir in root_dir.iterdir():
    if not case_dir.is_dir():
        continue

    for stage_dir in case_dir.iterdir():
        if not stage_dir.is_dir():
            continue

        # 例如 1prenatal -> prenatal
        phase = re.sub(r"^\d+", "", stage_dir.name)

        for visit_dir in stage_dir.iterdir():
            if not visit_dir.is_dir():
                continue

            visit_id = visit_dir.name

            # 从 visit_id 前8位提取日期
            if len(visit_id) >= 8 and visit_id[:8].isdigit():
                exam_date = f"{visit_id[:4]}-{visit_id[4:6]}-{visit_id[6:8]}"
            else:
                exam_date = ""

            meta_data = {
                "visit_id": visit_id,
                "phase": phase,
                "gestational_age_days": 0,
                "exam_date": exam_date
            }

            meta_path = visit_dir / "meta.json"
            report_path = visit_dir / "report.txt"

            # 处理 meta.json：只有空文件或 {} 才写入
            if is_empty_meta_json(meta_path):
                with open(meta_path, "w", encoding="utf-8") as f:
                    json.dump(meta_data, f, ensure_ascii=False, indent=2)
                print(f"已写入 meta.json: {meta_path}")
                created_meta_count += 1
            else:
                print(f"跳过 meta.json（已有有效内容）: {meta_path}")
                skipped_meta_count += 1

            # 处理 report.txt：只有真正空文件才创建/保留
            if is_empty_text_file(report_path):
                if not report_path.exists():
                    report_path.write_text("", encoding="utf-8")
                    print(f"已创建 report.txt: {report_path}")
                else:
                    print(f"report.txt 为空，跳过覆盖: {report_path}")
                created_report_count += 1
            else:
                print(f"跳过 report.txt（已有内容）: {report_path}")
                skipped_report_count += 1

print("\n处理完成")
print(f"写入 meta.json 数量: {created_meta_count}")
print(f"处理空 report.txt 数量: {created_report_count}")
print(f"跳过 meta.json 数量: {skipped_meta_count}")
print(f"跳过 report.txt 数量: {skipped_report_count}")


对应表格和report的内容，自动填入report
以case和一个表格为单位

只处理“检查文件夹日期”和“Excel 检查时间日期”可以一一对应的情况。

以下情况全部不写入 report.txt，只记录原因：

1. case 文件夹中存在多个前 8 位日期相同的检查文件夹
2. xlsx 表格中存在多个日期相同的检查记录
3. case 中有检查文件夹，但 xlsx 中没有对应日期
4. xlsx 中有检查记录，但 case 中没有对应检查文件夹

In [ ]:
from pathlib import Path
from datetime import datetime
from collections import defaultdict
import pandas as pd


# =========================
# 1. 输入路径
# =========================

case_dir = Path(r"G:\wxy_codes\Fetal_hydronephrosis\第二批数据及处理过程\batch2data_2\raw\case_0023")

xlsx_path = Path(r"G:\wxy_codes\Fetal_hydronephrosis\第二批数据及处理过程\2-阳性数据（新）包含图像和报告 260402\11林诗奇&牟潇杨.xls")


# =========================
# 2. 基本检查
# =========================

if not case_dir.exists():
    raise FileNotFoundError(f"case 文件夹不存在: {case_dir}")

if not xlsx_path.exists():
    raise FileNotFoundError(f"xlsx 文件不存在: {xlsx_path}")


# =========================
# 3. 读取 Excel
# =========================

df = pd.read_excel(xlsx_path)

print("Excel 列名：")
print(df.columns.tolist())

required_cols = ["检查时间", "描述"]

for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Excel 中缺少必要列: {col}")


# =========================
# 4. 解析 Excel 日期
# =========================

def parse_exam_date(value):
    """
    将 Excel 中的检查时间解析为 YYYYMMDD。
    例如：
    2025/6/6 12:36:42 -> 20250606
    """
    if pd.isna(value):
        return None

    try:
        dt = pd.to_datetime(value, errors="coerce")
    except Exception:
        return None

    if pd.isna(dt):
        return None

    return dt.strftime("%Y%m%d")


df["检查日期_YYYYMMDD"] = df["检查时间"].apply(parse_exam_date)


# =========================
# 5. 找到 case 下所有检查文件夹
# =========================

def is_exam_folder(path):
    """
    判断是否为检查文件夹：
    文件夹名至少 8 位，且前 8 位是合法日期。
    例如：
    202506061036
    202506251980
    """
    if not path.is_dir():
        return False

    name = path.name

    if len(name) < 8:
        return False

    date_part = name[:8]

    if not date_part.isdigit():
        return False

    try:
        datetime.strptime(date_part, "%Y%m%d")
        return True
    except Exception:
        return False


exam_dirs = [p for p in case_dir.rglob("*") if is_exam_folder(p)]

print(f"\n共找到 {len(exam_dirs)} 个检查文件夹。")


# =========================
# 6. 建立 case 日期 -> 检查文件夹 映射
# =========================

case_date_to_dirs = defaultdict(list)

for exam_dir in exam_dirs:
    date_key = exam_dir.name[:8]
    case_date_to_dirs[date_key].append(exam_dir)


# =========================
# 7. 建立 Excel 日期 -> 行记录 映射
# =========================

excel_date_to_rows = defaultdict(list)
invalid_excel_rows = []

for idx, row in df.iterrows():
    date_key = row["检查日期_YYYYMMDD"]

    # 关键修改：同时排除 None、NaN、NaT
    if pd.isna(date_key):
        invalid_excel_rows.append({
            "row_index": idx,
            "检查时间": row.get("检查时间", ""),
            "原因": "Excel 检查时间为空或无法解析"
        })
        continue

    date_key = str(date_key).strip()

    if date_key == "" or date_key.lower() in ["nat", "nan", "none"]:
        invalid_excel_rows.append({
            "row_index": idx,
            "检查时间": row.get("检查时间", ""),
            "原因": "Excel 检查时间为空或无法解析"
        })
        continue

    excel_date_to_rows[date_key].append(row)


# =========================
# 8. 区分可一一对应和异常情况
# =========================

all_case_dates = set(case_date_to_dirs.keys())
all_excel_dates = set(excel_date_to_rows.keys())
all_dates = sorted(all_case_dates | all_excel_dates)

matched_count = 0

skipped_records = []
written_records = []

for date_key in all_dates:
    case_dirs = case_date_to_dirs.get(date_key, [])
    excel_rows = excel_date_to_rows.get(date_key, [])

    # 情况1：case 中没有这个日期的检查文件夹
    if len(case_dirs) == 0 and len(excel_rows) > 0:
        skipped_records.append({
            "date": date_key,
            "reason": "Excel 中存在该日期记录，但 case 文件夹中没有对应检查文件夹",
            "case_dirs": "",
            "excel_row_count": len(excel_rows)
        })
        continue

    # 情况2：Excel 中没有这个日期的记录
    if len(case_dirs) > 0 and len(excel_rows) == 0:
        skipped_records.append({
            "date": date_key,
            "reason": "case 文件夹中存在该日期检查文件夹，但 Excel 中没有对应记录",
            "case_dirs": " | ".join(str(p) for p in case_dirs),
            "excel_row_count": 0
        })
        continue

    # 情况3：case 中同一天有多个检查文件夹
    if len(case_dirs) > 1:
        skipped_records.append({
            "date": date_key,
            "reason": "case 文件夹中存在多个前 8 位日期相同的检查文件夹，无法一一对应",
            "case_dirs": " | ".join(str(p) for p in case_dirs),
            "excel_row_count": len(excel_rows)
        })
        continue

    # 情况4：Excel 中同一天有多条记录
    if len(excel_rows) > 1:
        skipped_records.append({
            "date": date_key,
            "reason": "Excel 中存在多条日期相同的检查记录，无法一一对应",
            "case_dirs": " | ".join(str(p) for p in case_dirs),
            "excel_row_count": len(excel_rows)
        })
        continue

    # 情况5：一一对应，执行写入
    exam_dir = case_dirs[0]
    row = excel_rows[0]

    desc = row["描述"]

    if pd.isna(desc):
        desc = ""
    else:
        desc = str(desc).strip()

    report_path = exam_dir / "report.txt"

    with open(report_path, "w", encoding="utf-8") as f:
        f.write(desc)

    matched_count += 1

    written_records.append({
        "date": date_key,
        "exam_dir": str(exam_dir),
        "report_path": str(report_path),
        "检查时间": row["检查时间"]
    })

    print(f"已写入: {report_path}")


# =========================
# 9. 整理异常 Excel 行
# =========================

for item in invalid_excel_rows:
    skipped_records.append({
        "date": "",
        "reason": item["原因"],
        "case_dirs": "",
        "excel_row_count": "",
        "检查时间": item["检查时间"],
        "excel_row_index": item["row_index"]
    })


# =========================
# 10. 保存处理日志
# =========================

log_dir = case_dir / "_match_logs"
log_dir.mkdir(exist_ok=True)

written_log_path = log_dir / "written_records.csv"
skipped_log_path = log_dir / "skipped_records.csv"

pd.DataFrame(written_records).to_csv(
    written_log_path,
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame(skipped_records).to_csv(
    skipped_log_path,
    index=False,
    encoding="utf-8-sig"
)


# =========================
# 11. 输出汇总
# =========================

print("\n处理完成。")
print(f"成功一一匹配并写入 report.txt 的数量: {matched_count}")
print(f"跳过未处理的异常记录数量: {len(skipped_records)}")

print(f"\n成功写入日志已保存到:")
print(written_log_path)

print(f"\n跳过记录日志已保存到:")
print(skipped_log_path)

if skipped_records:
    print("\n以下是跳过记录预览：")
    for item in skipped_records[:20]:
        print(item)